# Mutual Funds Analytics - Interactive Exploration

This notebook guides you through fetching, cleaning, analyzing, and plotting Mutual Fund historical Net Asset Values (NAV) in Python.

In [ ]:
import os
import requests
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Setting plotting styles
plt.style.use('seaborn-v0_8-darkgrid')
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['figure.dpi'] = 100

## 1. Fetch Mutual Fund Data

We use the free public API `https://api.mfapi.in/mf` to fetch NAV data. For example, let's fetch data for **Parag Parikh Flexi Cap Fund - Direct Plan** (Scheme Code: `122639`).

In [ ]:
scheme_code = 122639
url = f"https://api.mfapi.in/mf/{scheme_code}"

print(f"Fetching data from: {url}")
response = requests.get(url)
data = response.json()

meta = data['meta']
nav_data = data['data']

print("\n--- Scheme Metadata ---")
for key, val in meta.items():
    print(f"{key.replace('_', ' ').title()}: {val}")

## 2. Load into Pandas & Clean Data

Convert the JSON data array into a Pandas DataFrame and structure it with appropriate data types.

In [ ]:
df = pd.DataFrame(nav_data)
df.rename(columns={'date': 'Date', 'nav': 'NAV'}, inplace=True)

# Convert types
df['Date'] = pd.to_datetime(df['Date'], format='%d-%m-%Y')
df['NAV'] = pd.to_numeric(df['NAV'])

# Chronological sort
df.sort_values('Date', inplace=True)
df.reset_index(drop=True, inplace=True)

print(f"Dataset shape: {df.shape}")
df.head()

## 3. Performance Metrics calculation

Let's compute the daily returns and the moving averages (30-day and 90-day).

In [ ]:
# Rolling return & volatility calculations
df['Daily_Return'] = df['NAV'].pct_change()
df['Rolling_30_NAV'] = df['NAV'].rolling(window=30, min_periods=1).mean()
df['Rolling_90_NAV'] = df['NAV'].rolling(window=90, min_periods=1).mean()

# 30-day rolling annualized volatility (252 trading days/year)
df['Rolling_Vol_30d'] = df['Daily_Return'].rolling(window=30, min_periods=1).std() * np.sqrt(252)

df.tail()

## 4. Visualizations

### Plot NAV and Moving Averages

In [ ]:
plt.figure(figsize=(14, 7))
plt.plot(df['Date'], df['NAV'], label='Latest NAV', color='#2ecc71', lw=2)
plt.plot(df['Date'], df['Rolling_30_NAV'], label='30-day SMA', color='#e67e22', linestyle='--')
plt.plot(df['Date'], df['Rolling_90_NAV'], label='90-day SMA', color='#3498db', linestyle=':')
plt.title(f"{meta.get('scheme_name')} - NAV Trend", fontsize=14, fontweight='bold')
plt.xlabel('Date')
plt.ylabel('NAV (INR)')
plt.legend(loc='upper left')
plt.tight_layout()
plt.show()

### Plot Volatility Distribution & Timeline

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

# Histogram of daily returns
sns.histplot(df['Daily_Return'].dropna(), bins=60, color='teal', kde=True, ax=ax1)
ax1.set_title('Distribution of Daily Returns')
ax1.set_xlabel('Daily Return')

# Rolling 30-day volatility
ax2.plot(df['Date'], df['Rolling_Vol_30d'] * 100, color='crimson')
ax2.set_title('Annualized 30-Day Rolling Volatility')
ax2.set_ylabel('Volatility (%)')
ax2.set_xlabel('Date')

plt.tight_layout()
plt.show()